In [1]:

from common import q  # Importing 'q' from the 'common' module (assumed to be some mathematical or computational utility).
from numpy import set_printoptions  # Importing 'set_printoptions' from NumPy to control how NumPy arrays are displayed.
from math import sqrt  # Importing 'sqrt' from the math module for square root operations.
from fft import fft, ifft, sub, neg, add_fft, mul_fft  # Importing various functions from the 'fft' module for Fourier Transform operations.
from ntt import sub_zq, mul_zq, div_zq  # Importing polynomial arithmetic operations over the cyclotomic ring from 'ntt' module.
from ffsampling import gram, ffldl_fft, ffsampling_fft  # Importing sampling-related functions from 'ffsampling' module.
from ntrugen import ntru_gen  # Importing 'ntru_gen' from 'ntrugen' for NTRU key generation.
from encoding import compress, decompress  # Importing 'compress' and 'decompress' functions from the 'encoding' module for data compression.
# Importing the SHAKE256 hash function from PyCryptodome for cryptographic operations.
from Crypto.Hash import SHAKE256

# Importing 'urandom' from the 'os' module for generating random bytes.
from os import urandom
# Importing the ChaCha20 random number generator from the 'rng' module.
from rng import ChaCha20
# For debugging purposes
import sys

# Checking Python version to determine if the 'reload' function should be imported.
if sys.version_info >= (3, 4):  
    from importlib import reload  # Import 'reload' for re-importing modules if using Python 3.4 or later.

# Setting display options for NumPy arrays
set_printoptions(linewidth=200, precision=5, suppress=True)  # Configuring how arrays are printed: set the maximum line width and precision.

# Dictionary defining different parameter sets for Falcon cryptographic algorithm.
logn = {  # Dictionary mapping the log base 2 of parameter 'n'.
    2: 1,
    4: 2,
    8: 3,
    16: 4,
    32: 5,
    64: 6,
    128: 7,
    256: 8,
    512: 9,
    1024: 10
}

# Constants for byte lengths of different cryptographic components.
HEAD_LEN = 1  # Length of the signing salt and header in bytes.
SALT_LEN = 50  # Length of the salt in bytes.
SEED_LEN = 56  # Length of the seed in bytes.

# Parameter sets for Falcon:
# - n: dimension/degree of the cyclotomic ring
# - sigma: standard deviation of the signatures (Gaussians over a lattice)
# - sigmin: lower bound on the standard deviation of each Gaussian over Z
# - sigbound: upper bound on the squared norm ||s0||^2 + ||s1||^2
# - sig_bytelen: byte length of the signatures
Params = {
    # FalconParam(2, 2)
    2: {  # Parameters for Falcon with n=2
        "n": 2,
        "sigma": 144.81253976308423,
        "sigmin": 1.1165085072329104,
        "sig_bound": 101498,
        "sig_bytelen": 44,
    },
    # FalconParam(4, 2)
    4: {  # Parameters for Falcon with n=4
        "n": 4,
        "sigma": 146.83798833523608,
        "sigmin": 1.1321247692325274,
        "sig_bound": 208714,
        "sig_bytelen": 47,
    },
    # FalconParam(8, 2)
    8: {  # Parameters for Falcon with n=8
        "n": 8,
        "sigma": 148.83587593064718,
        "sigmin": 1.147528535373367,
        "sig_bound": 428865,
        "sig_bytelen": 52,
    },
    # FalconParam(16, 4)
    16: {  # Parameters for Falcon with n=16
        "n": 16,
        "sigma": 151.78340713845503,
        "sigmin": 1.170254078853483,
        "sig_bound": 892039,
        "sig_bytelen": 63,
    },
    # FalconParam(32, 8)
    32: {  # Parameters for Falcon with n=32
        "n": 32,
        "sigma": 154.6747794602761,
        "sigmin": 1.1925466358390344,
        "sig_bound": 1852696,
        "sig_bytelen": 82,
    },
    # FalconParam(64, 16)
    64: {  # Parameters for Falcon with n=64
        "n": 64,
        "sigma": 157.51308555044122,
        "sigmin": 1.2144300507766141,
        "sig_bound": 3842630,
        "sig_bytelen": 122,
    },
    # FalconParam(128, 32)
    128: {  # Parameters for Falcon with n=128
        "n": 128,
        "sigma": 160.30114421975344,
        "sigmin": 1.235926056771981,
        "sig_bound": 7959734,
        "sig_bytelen": 200,
    },
    # FalconParam(256, 64)
    256: {  # Parameters for Falcon with n=256
        "n": 256,
        "sigma": 163.04153322607107,
        "sigmin": 1.2570545284063217,
        "sig_bound": 16468416,
        "sig_bytelen": 356,
    },
    # FalconParam(512, 128)
    512: {  # Parameters for Falcon with n=512
        "n": 512,
        "sigma": 165.7366171829776,
        "sigmin": 1.2778336969128337,
        "sig_bound": 34034726,
        "sig_bytelen": 666,
    },
    # FalconParam(1024, 256)
    1024: {  # Parameters for Falcon with n=1024
        "n": 1024,
        "sigma": 168.38857144654395,
        "sigmin": 1.298280334344292,
        "sig_bound": 70265242,
        "sig_bytelen": 1280,
    },
}



def print_tree(tree, pref=""):
    """
    Display a LDL tree in a readable form.

    Args:
        tree: A LDL tree (represented as a list)
        pref: A string prefix for indentation, used to control the display format

    Format: coefficient or fft

    Returns:
        A string representation of the LDL tree in a readable format.
    """
    # Define symbols for different levels of the tree display
    leaf = "|_____> "  # Leaf node symbol
    top = "|_______"  # Top-level node symbol
    son1 = "|       "  # Symbol for the first child
    son2 = "        "  # Symbol for the second child
    width = len(top)  # Width used for indentation based on tree depth

    # Initialize the output string
    a = ""
    
    # Check if the current node is a tree (3 elements: [value, left, right])
    if len(tree) == 3:
        if pref == "":
            a += pref + str(tree[0]) + "\n"  # Display root value if no prefix
        else:
            a += pref[:-width] + top + str(tree[0]) + "\n"  # Display top level node with value
        
        # Recursively print left and right subtrees with appropriate indentation
        a += print_tree(tree[1], pref + son1)  # Print left child
        a += print_tree(tree[2], pref + son2)  # Print right child
        
        return a  # Return the accumulated string

    else:
        return (pref[:-width] + leaf + str(tree) + "\n")  # Leaf node representation


def normalize_tree(tree, sigma):
    """
    Normalize leaves of a LDL tree (from values ||b_i||**2 to sigma/||b_i||).

    Args:
        tree: A LDL tree (represented as a list)
        sigma: A standard deviation used for normalization

    Format: coefficient or fft

    Returns:
        A tree where leaf nodes are normalized by sigma / ||b_i||
    """
    # Check if the current node is a tree (3 elements: [value, left, right])
    if len(tree) == 3:
        normalize_tree(tree[1], sigma)  # Recursively normalize left subtree
        normalize_tree(tree[2], sigma)  # Recursively normalize right subtree
    else:
        # Normalize leaf node from ||b_i||**2 to sigma / ||b_i||
        tree[0] = sigma / sqrt(tree[0].real)  # Normalize by sigma over the real part of ||b_i||
        tree[1] = 0  # Reset imaginary part to 0 after normalization


class PublicKey:
    """
    This class contains methods for performing public key operations in Falcon.
    """

    def __init__(self, sk):
        """
        Initialize a public key.

        Args:
            sk: SecretKey object used to initialize the public key from private parameters.
        """
        self.n = sk.n  # Dimension/degree of the cyclotomic ring
        self.h = sk.h  # The polynomial h such that h*f = g mod (Phi, q)
        self.hash_to_point = sk.hash_to_point  # Method for hashing a message to a point
        self.signature_bound = sk.signature_bound  # Bound on signature norm
        self.verify = sk.verify  # Method for verifying a signature

    def __repr__(self):
        """
        Print the object in a readable form.

        Returns:
            A string representation of the public key.
        """
        rep = "Public for n = {n}:\n\n".format(n=self.n)
        rep += "h = {h}\n".format(h=self.h)
        return rep



class SecretKey:
    """
    This class contains methods for performing secret key operations (and also public key operations) in Falcon.
    
    One can:
    - initialize a secret key for:
        - n = 128, 256, 512, 1024,
        - phi = x ** n + 1,
        - q = 12 * 1024 + 1
    - find a preimage t of a point c (both in ( Z[x] mod (Phi,q) )**2 ) such that t*B0 = c
    - hash a message to a point of Z[x] mod (Phi,q)
    - sign a message
    - verify the signature of a message
    """

    def __init__(self, n, polys=None):
        """
        Initialize a secret key.

        Args:
            n: Dimension/degree of the cyclotomic ring.
            polys: Optional list of NTRU polynomials (f, g, F, G).
        """
        # Public parameters from Params dictionary
        self.n = n
        self.sigma = Params[n]["sigma"]
        self.sigmin = Params[n]["sigmin"]
        self.signature_bound = Params[n]["sig_bound"]
        self.sig_bytelen = Params[n]["sig_bytelen"]

        # Generate NTRU polynomials f, g, F, G if not provided
        if polys is None:
            self.f, self.g, self.F, self.G = ntru_gen(n)
        else:
            [f, g, F, G] = polys
            assert all((len(poly) == n) for poly in [f, g, F, G])
            self.f = f[:]
            self.g = g[:]
            self.F = F[:]
            self.G = G[:]

        # Compute the basis B0 of a NTRU lattice from f, g, F, G
        B0 = [[self.g, neg(self.f)], [self.G, neg(self.F)]]
        G0 = gram(B0)
        self.B0_fft = [[fft(elt) for elt in row] for row in B0]  # FFT of B0
        G0_fft = [[fft(elt) for elt in row] for row in G0]  # FFT of Gram matrix

        # LDL factorization of the Gram matrix
        self.T_fft = ffldl_fft(G0_fft)

        # Normalize the Falcon tree based on sigma
        normalize_tree(self.T_fft, self.sigma)

        # The public key is derived from g and f
        self.h = div_zq(self.g, self.f)

    def __repr__(self, verbose=False):
        """
        Print the object in a readable form.

        Args:
            verbose: If `True`, include the FFT tree.

        Returns:
            A string representation of the secret key, including polynomials.
        """
        rep = "Private key for n = {n}:\n\n".format(n=self.n)
        rep += "f = {f}\n".format(f=self.f)
        rep += "g = {g}\n".format(g=self.g)
        rep += "F = {F}\n".format(F=self.F)
        rep += "G = {G}\n".format(G=self.G)
        if verbose:
            rep += "\nFFT tree\n"
            rep += print_tree(self.T_fft, pref="")
        return rep

    def hash_to_point(self, message, salt):
        """
        Hash a message to a point in Z[x] mod(Phi, q).
        
        Args:
            message: The message to be hashed.
            salt: A salt value added during hashing.

        Returns:
            A list of coefficients in Z[x] mod(Phi, q) representing the hashed message.
        """
        n = self.n
        if q > (1 << 16):  # Check if modulus is too large
            raise ValueError("The modulus is too large")

        k = (1 << 16) // q  # Constant used for rejection sampling
        shake = SHAKE256.new()
        shake.update(salt)
        shake.update(message)

        hashed = [0 for i in range(n)]  # Result array of coefficients
        i = 0
        while i < n:
            # Read two bytes and transform them into a 16-bit integer
            twobytes = shake.read(2)
            elt = (twobytes[0] << 8) + twobytes[1]  # Merge bytes into 16-bit int
            if elt < k * q:  # Implicit rejection sampling
                hashed[i] = elt % q
                i += 1
        return hashed

    def sample_preimage(self, point, seed=None):
        """
        Sample a short vector s such that s[0] + s[1] * h = point.

        Args:
            point: A point in Z[x] mod(Phi, q) to find a preimage for.
            seed: Optional seed for generating randomness (used for PRG).

        Returns:
            A vector s = (s0, s1) such that s0 + s1 * h ≈ point.
        """
        [[a, b], [c, d]] = self.B0_fft

        point_fft = fft(point)  # FFT of the input point
        t0_fft = [(point_fft[i] * d[i]) / q for i in range(self.n)]
        t1_fft = [(-point_fft[i] * b[i]) / q for i in range(self.n)]
        t_fft = [t0_fft, t1_fft]  # The transformed point

        if seed is None:
            # Use urandom as the pseudo-random source
            z_fft = ffsampling_fft(t_fft, self.T_fft, self.sigmin, urandom)
        else:
            # Use ChaCha20 PRG for pseudo-randomness
            chacha_prng = ChaCha20(seed)
            z_fft = ffsampling_fft(t_fft, self.T_fft, self.sigmin, chacha_prng.randombytes)

        v0_fft = add_fft(mul_fft(z_fft[0], a), mul_fft(z_fft[1], c))
        v1_fft = add_fft(mul_fft(z_fft[0], b), mul_fft(z_fft[1], d))
        v0 = [int(round(elt)) for elt in ifft(v0_fft)]
        v1 = [int(round(elt)) for elt in ifft(v1_fft)]

        s = [sub(point, v0), neg(v1)]
        return s

    def sign(self, message, randombytes=urandom):
        """
        Sign a message using the secret key.

        Args:
            message: The message to be signed.
            randombytes: Optional source of randomness (default is urandom).

        Returns:
            A signature that consists of a header, salt, and encoded signature.
        """
        int_header = 0x30 + logn[self.n]  # Construct the header byte based on the key size
        header = int_header.to_bytes(1, "little")

        salt = randombytes(SALT_LEN)  # Generate a random salt
        hashed = self.hash_to_point(message, salt)  # Hash the message to a point

        while True:
            if randombytes == urandom:
                s = self.sample_preimage(hashed)  # Sample a preimage using urandom
            else:
                seed = randombytes(SEED_LEN)
                s = self.sample_preimage(hashed, seed=seed)

            norm_sign = sum(coef ** 2 for coef in s[0]) + sum(coef ** 2 for coef in s[1])  # Calculate the norm

            if norm_sign <= self.signature_bound:  # Check if the norm of the signature is within bounds
                enc_s = compress(s[1], self.sig_bytelen - HEAD_LEN - SALT_LEN)  # Compress the signature
                if enc_s is not False:
                    return header + salt + enc_s

    def verify(self, message, signature):
        """
        Verify a signature.

        Args:
            message: The message to be verified.
            signature: The signature to verify.

        Returns:
            True if the signature is valid, otherwise False.
        """
        salt = signature[HEAD_LEN:HEAD_LEN + SALT_LEN]  # Extract the salt from the signature
        enc_s = signature[HEAD_LEN + SALT_LEN:]  # Extract the encoded signature

        s1 = decompress(enc_s, self.sig_bytelen - HEAD_LEN - SALT_LEN, self.n)  # Decode the signature

        if s1 is False:
            print("Invalid encoding")
            return False

        hashed = self.hash_to_point(message, salt)  # Hash the message to a point
        s0 = sub_zq(hashed, mul_zq(s1, self.h))  # Calculate s0 = hashed - s1 * h

        # Normalize coefficients in (-q/2, q/2]
        s0 = [(coef + (q >> 1)) % q - (q >> 1) for coef in s0]

        norm_sign = sum(coef ** 2 for coef in s0) + sum(coef ** 2 for coef in s1)  # Calculate the norm of the signature
        if norm_sign > self.signature_bound:
            print("Squared norm of signature is too large:", norm_sign)
            return False

        return True


In [2]:
%%time
import pandas as pd
from io import BytesIO

# Define the FalconDataFrameHandler class
class FalconDataFrameHandler:
    
    def __init__(self, secret_key):
        """
        Initialize the FalconDataFrameHandler with a secret key.
        
        Args:
            secret_key: An instance of the SecretKey class for encryption/decryption.
        """
        self.sk = secret_key  # Store the secret key
        self.pk = PublicKey(secret_key)  # Generate the corresponding public key from the secret key

    def encrypt_dataframe(self, df):
        """
        Encrypt a DataFrame using the Falcon signature mechanism.
        
        Args:
            df: pandas DataFrame to encrypt.
        
        Returns:
            dict with 'encrypted_data' containing the encrypted DataFrame bytes and 'signature' containing the signature.
        """
        # Serialize the DataFrame to bytes
        df_bytes = BytesIO()
        df.to_pickle(df_bytes)  # Use pandas' pickle method to convert DataFrame to bytes
        df_bytes = df_bytes.getvalue()  # Get the byte representation
        
        # Encrypt by signing the serialized DataFrame
        signature = self.sk.sign(df_bytes)  # Sign the DataFrame bytes using the secret key

        return {"encrypted_data": df_bytes, "signature": signature}  # Return the encrypted data and signature

    def decrypt_dataframe(self, encrypted_data):
        """
        Decrypt a DataFrame using the Falcon verification mechanism.
        
        Args:
            encrypted_data: A dictionary containing 'encrypted_data' (the DataFrame bytes) and 'signature'.
        
        Returns:
            The decrypted pandas DataFrame if verification succeeds, else raises a ValueError.
        """
        encrypted_bytes = encrypted_data["encrypted_data"]  # Extract the encrypted DataFrame bytes
        signature = encrypted_data["signature"]  # Extract the signature

        # Verify the signature
        if self.pk.verify(encrypted_bytes, signature):  # Verify using the public key
            # Deserialize the bytes back to DataFrame
            df = pd.read_pickle(BytesIO(encrypted_bytes))  # Convert bytes back to DataFrame
            return df  # Return the decrypted DataFrame
        else:
            raise ValueError("Signature verification failed. Data may be corrupted.")  # Raise an error if verification fails

# Example Usage
if __name__ == "__main__":
    n = 256  # Key size parameter

    if n in Params:  # Check if 'n' is in the defined parameters (likely from Falcon settings)
        # Initialize the Falcon Secret Key
        sk = SecretKey(n)  # Create a secret key using the specified parameter 'n'

        # Create a handler object for encryption/decryption
        falcon_handler = FalconDataFrameHandler(sk)

        # Load or define an example DataFrame
        df = pd.read_csv("elliptic_txs_features.csv")  # Load a DataFrame from a CSV file

        # Create a subset of the DataFrame for encryption
        df2 = df.sample(frac=0.2, random_state=42)  # Sample 20% of the DataFrame for testing

        print("Original DataFrame:", df2)  # Print the sample DataFrame

        # Encrypt the sample DataFrame
        encrypted = falcon_handler.encrypt_dataframe(df2)  # Perform encryption
        print("Encrypted Data:", encrypted)  # Print the encrypted data and signature

        # Decrypt the DataFrame
        decrypted_df = falcon_handler.decrypt_dataframe(encrypted)  # Perform decryption
        print("Decrypted DataFrame:")
        print(decrypted_df)  # Print the decrypted DataFrame

    else:
        print("Key not in parameter")  # If the parameter 'n' is not defined in the configuration


Original DataFrame:         230425980   1  -0.1714692896288031  -0.18466755143291433  \
107272  272481041  24            -0.154168             -0.107012   
103928  331556016  23            -0.063761             -0.158783   
150374   29913073  37            -0.030351              0.138895   
89566   209062970  20            -0.172982             -0.175507   
159898   97388854  40            -0.160374              0.072863   
...           ...  ..                  ...                   ...   
157441   98034907  40            -0.171422             -0.035052   
50163    21974167   9            -0.169838             -0.191594   
103872  141371446  23            -0.116215             -0.173599   
110291  275521192  25            -0.170933             -0.109795   
202709  158292105  49            -0.172513             -0.092657   

        -1.2013688016765636  -0.12196959975910057  -0.04387454791734898  \
107272            -1.201369             -0.121970             -0.043875   
103928       

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Decrypted DataFrame:
        230425980   1  -0.1714692896288031  -0.18466755143291433  \
107272  272481041  24            -0.154168             -0.107012   
103928  331556016  23            -0.063761             -0.158783   
150374   29913073  37            -0.030351              0.138895   
89566   209062970  20            -0.172982             -0.175507   
159898   97388854  40            -0.160374              0.072863   
...           ...  ..                  ...                   ...   
157441   98034907  40            -0.171422             -0.035052   
50163    21974167   9            -0.169838             -0.191594   
103872  141371446  23            -0.116215             -0.173599   
110291  275521192  25            -0.170933             -0.109795   
202709  158292105  49            -0.172513             -0.092657   

        -1.2013688016765636  -0.12196959975910057  -0.04387454791734898  \
107272            -1.201369             -0.121970             -0.043875   
103928      